# Proposed pipeline (v2 — label-mapping fix)

Same TF-IDF + Chi2/MI feature selection + GridSearchCV pipeline as
`proposed_improvements.ipynb`, with the label-mapping bug fixed (I-1) and a corrected
evaluation: GridSearchCV now optimizes macro-F1 instead of positive-class-only F1, and
results report macro-F1, per-class precision/recall/F1, and a confusion matrix for every
model (I-2). `class_weight='balanced'` is added as a tunable option for LR/SVM (I-5).
Results are merged with the baseline + Dummy results from `baseline_pipeline_v2.ipynb`
into one comparable table (I-4). SHAP explainability is deferred to a later pass once the
corrected numbers are the ones being explained (see ISSUE_PLAN.md Phase 5).

In [1]:
import re

import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

from liar_utils import RANDOM_STATE, evaluate_full, load_and_label, print_report

ModuleNotFoundError: No module named 'pandas'

Load data with the corrected label mapping

In [ ]:
train = load_and_label("train.csv")
valid = load_and_label("valid.csv")
test = load_and_label("test.csv")

balance = train["Label"].value_counts(normalize=True).rename({0: "fake", 1: "real"})
print("Train class balance:\n", balance)
assert 0.35 < balance["fake"] < 0.5, "Fake-class share outside the expected ~44% range -- check label mapping"

Train class balance:
 Label
real    0.561719
fake    0.438281
Name: proportion, dtype: float64


Text preprocessing (stopword removal + stemming)

In [ ]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return " ".join(words)


train["clean_text"] = train["Statement"].apply(preprocess)
valid["clean_text"] = valid["Statement"].apply(preprocess)
test["clean_text"] = test["Statement"].apply(preprocess)

TF-IDF feature extraction

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.9)

X_train_tfidf = tfidf.fit_transform(train["clean_text"])
X_valid_tfidf = tfidf.transform(valid["clean_text"])
X_test_tfidf = tfidf.transform(test["clean_text"])

y_train = train["Label"]
y_valid = valid["Label"]
y_test = test["Label"]

print("TF-IDF shape:", X_train_tfidf.shape)

TF-IDF shape: (10240, 10000)


Chi-Square feature selection

In [ ]:
k_features = 3000

chi2_selector = SelectKBest(score_func=chi2, k=k_features)

X_train_chi2 = chi2_selector.fit_transform(X_train_tfidf, y_train)
X_valid_chi2 = chi2_selector.transform(X_valid_tfidf)
X_test_chi2 = chi2_selector.transform(X_test_tfidf)

print("Chi-square selected shape:", X_train_chi2.shape)

Chi-square selected shape: (10240, 3000)


Mutual Information feature selection

In [ ]:
mi_selector = SelectKBest(score_func=lambda X, y: mutual_info_classif(X, y, random_state=RANDOM_STATE), k=k_features)

X_train_mi = mi_selector.fit_transform(X_train_tfidf, y_train)
X_valid_mi = mi_selector.transform(X_valid_tfidf)
X_test_mi = mi_selector.transform(X_test_tfidf)

print("Mutual Information selected shape:", X_train_mi.shape)

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received 

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received 

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received 

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received 

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received 

...

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received 

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received 

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received 

Mutual Information selected shape: (10240, 3000)


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received 

GridSearch + evaluation -- macro-F1 is now the tuning objective (I-2)

In [ ]:
def train_and_evaluate(model, param_grid, X_train, y_train, X_valid, y_valid, X_test, y_test):
    grid = GridSearchCV(model, param_grid, cv=5, scoring="f1_macro", n_jobs=-1)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    valid_metrics = evaluate_full(y_valid, best_model.predict(X_valid))
    test_metrics = evaluate_full(y_test, best_model.predict(X_test))

    return best_model, grid.best_params_, valid_metrics, test_metrics

Model + param-grid definitions (class_weight='balanced' added to LR/SVM per I-5)

In [ ]:
def make_models():
    return [
        (
            "Logistic Regression",
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            {"C": [0.1, 1, 10], "solver": ["liblinear"], "class_weight": [None, "balanced"]},
        ),
        (
            "SVM",
            LinearSVC(random_state=RANDOM_STATE),
            {"C": [0.1, 1, 10], "class_weight": [None, "balanced"]},
        ),
        ("Naive Bayes", MultinomialNB(), {"alpha": [0.1, 0.5, 1.0]}),
        (
            "Random Forest",
            RandomForestClassifier(random_state=RANDOM_STATE),
            {
                "n_estimators": [100, 200],
                "max_depth": [None, 10],
                "min_samples_split": [2, 5],
            },
        ),
        (
            "XGBoost",
            XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
            {
                "n_estimators": [100, 200],
                "max_depth": [3, 6],
                "learning_rate": [0.01, 0.1],
            },
        ),
    ]

Run experiments (Chi² features)

In [ ]:
results_chi2 = {}
best_models_chi2 = {}

for name, model, params in make_models():
    print(f"\nTraining {name} with Chi-square features...")
    best_model, best_params, valid_metrics, test_metrics = train_and_evaluate(
        model, params, X_train_chi2, y_train, X_valid_chi2, y_valid, X_test_chi2, y_test
    )
    print("Best params:", best_params)
    print_report(name, y_test, best_model.predict(X_test_chi2))
    results_chi2[name] = {"best_params": best_params, "valid": valid_metrics, "test": test_metrics}
    best_models_chi2[name] = best_model


Training Logistic Regression with Chi-square features...


Best params: {'C': 10, 'class_weight': 'balanced', 'solver': 'liblinear'}

Logistic Regression
[[311 242]
 [272 442]]
              precision    recall  f1-score   support

        fake      0.533     0.562     0.548       553
        real      0.646     0.619     0.632       714

    accuracy                          0.594      1267
   macro avg      0.590     0.591     0.590      1267
weighted avg      0.597     0.594     0.595      1267


Training SVM with Chi-square features...
Best params: {'C': 10, 'class_weight': 'balanced'}

SVM
[[304 249]
 [288 426]]
              precision    recall  f1-score   support

        fake      0.514     0.550     0.531       553
        real      0.631     0.597     0.613       714

    accuracy                          0.576      1267
   macro avg      0.572     0.573     0.572      1267
weighted avg      0.580     0.576     0.577      1267


Training Naive Bayes with Chi-square features...
Best params: {'alpha': 0.1}

Naive Bayes
[[220 333]
 [170

Best params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}

Random Forest
[[259 294]
 [210 504]]
              precision    recall  f1-score   support

        fake      0.552     0.468     0.507       553
        real      0.632     0.706     0.667       714

    accuracy                          0.602      1267
   macro avg      0.592     0.587     0.587      1267
weighted avg      0.597     0.602     0.597      1267


Training XGBoost with Chi-square features...


Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}

XGBoost
[[180 373]
 [129 585]]
              precision    recall  f1-score   support

        fake      0.583     0.325     0.418       553
        real      0.611     0.819     0.700       714

    accuracy                          0.604      1267
   macro avg      0.597     0.572     0.559      1267
weighted avg      0.598     0.604     0.577      1267



Run experiments (MI features)

In [ ]:
results_mi = {}
best_models_mi = {}

for name, model, params in make_models():
    print(f"\nTraining {name} with MI features...")
    best_model, best_params, valid_metrics, test_metrics = train_and_evaluate(
        model, params, X_train_mi, y_train, X_valid_mi, y_valid, X_test_mi, y_test
    )
    print("Best params:", best_params)
    print_report(name, y_test, best_model.predict(X_test_mi))
    results_mi[name] = {"best_params": best_params, "valid": valid_metrics, "test": test_metrics}
    best_models_mi[name] = best_model


Training Logistic Regression with MI features...
Best params: {'C': 0.1, 'class_weight': 'balanced', 'solver': 'liblinear'}

Logistic Regression
[[340 213]
 [266 448]]
              precision    recall  f1-score   support

        fake      0.561     0.615     0.587       553
        real      0.678     0.627     0.652       714

    accuracy                          0.622      1267
   macro avg      0.619     0.621     0.619      1267
weighted avg      0.627     0.622     0.623      1267


Training SVM with MI features...


Best params: {'C': 0.1, 'class_weight': 'balanced'}

SVM
[[307 246]
 [268 446]]
              precision    recall  f1-score   support

        fake      0.534     0.555     0.544       553
        real      0.645     0.625     0.634       714

    accuracy                          0.594      1267
   macro avg      0.589     0.590     0.589      1267
weighted avg      0.596     0.594     0.595      1267


Training Naive Bayes with MI features...
Best params: {'alpha': 0.1}

Naive Bayes
[[221 332]
 [173 541]]
              precision    recall  f1-score   support

        fake      0.561     0.400     0.467       553
        real      0.620     0.758     0.682       714

    accuracy                          0.601      1267
   macro avg      0.590     0.579     0.574      1267
weighted avg      0.594     0.601     0.588      1267


Training Random Forest with MI features...


Best params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}

Random Forest
[[241 312]
 [185 529]]
              precision    recall  f1-score   support

        fake      0.566     0.436     0.492       553
        real      0.629     0.741     0.680       714

    accuracy                          0.608      1267
   macro avg      0.597     0.588     0.586      1267
weighted avg      0.601     0.608     0.598      1267


Training XGBoost with MI features...


Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}

XGBoost
[[205 348]
 [139 575]]
              precision    recall  f1-score   support

        fake      0.596     0.371     0.457       553
        real      0.623     0.805     0.703       714

    accuracy                          0.616      1267
   macro avg      0.609     0.588     0.580      1267
weighted avg      0.611     0.616     0.595      1267



Results to table, merged with the v2 baseline + Dummy results (I-4 exit criterion)

In [ ]:
def results_to_dataframe(results_dict, method_name):
    rows = []
    for model_name, data in results_dict.items():
        valid_m = data["valid"]
        test_m = data["test"]
        rows.append(
            {
                "Pipeline": "Proposed",
                "Method": method_name,
                "Model": model_name,
                "Valid Accuracy": valid_m["accuracy"],
                "Valid Macro-F1": valid_m["macro_f1"],
                "Valid Fake F1": valid_m["fake_f1"],
                "Test Accuracy": test_m["accuracy"],
                "Test Macro-F1": test_m["macro_f1"],
                "Test Fake Precision": test_m["fake_precision"],
                "Test Fake Recall": test_m["fake_recall"],
                "Test Fake F1": test_m["fake_f1"],
                "Test Real F1": test_m["real_f1"],
                "Test Confusion Matrix": test_m["confusion_matrix"],
                "Best Params": data["best_params"],
            }
        )
    return pd.DataFrame(rows)


df_chi2 = results_to_dataframe(results_chi2, "Chi-square")
df_mi = results_to_dataframe(results_mi, "Mutual Information")
proposed_results = pd.concat([df_chi2, df_mi], ignore_index=True)

baseline_results = pd.read_csv("baseline_results_v2.csv")

final_results = pd.concat([baseline_results, proposed_results], ignore_index=True)
final_results = final_results.sort_values("Test Macro-F1", ascending=False)
final_results

,Pipeline,Method,Model,Valid Accuracy,Valid Macro-F1,Valid Fake F1,Test Accuracy,Test Macro-F1,Test Fake Precision,Test Fake Recall,Test Fake F1,Test Real F1,Test Confusion Matrix,Best Params
13,Proposed,Mutual Information,Logistic Regression,0.626947,0.626929,0.624314,0.621942,0.619175,0.561056,0.614828,0.586713,0.651636,"[[340, 213], [266, 448]]","{'C': 0.1, 'class_weight': 'balanced', 'solver..."
3,Baseline,TF-IDF only,Logistic Regression,0.612928,0.603981,0.544455,0.621942,0.600735,0.587678,0.448463,0.508718,0.692752,"[[248, 305], [174, 540]]",NaN
4,Baseline,TF-IDF only,Logistic Regression (balanced),0.605919,0.605642,0.595200,0.602210,0.597778,0.542169,0.569620,0.555556,0.640000,"[[315, 238], [266, 448]]",NaN
5,Baseline,TF-IDF only,SVM,0.593458,0.591173,0.560606,0.602210,0.591301,0.548323,0.502712,0.524528,0.658073,"[[278, 275], [229, 485]]",NaN
8,Proposed,Chi-square,Logistic Regression,0.605919,0.605732,0.597134,0.594317,0.589934,0.533448,0.562387,0.547535,0.632332,"[[311, 242], [272, 442]]","{'C': 10, 'class_weight': 'balanced', 'solver'..."
14,Proposed,Mutual Information,SVM,0.609813,0.609555,0.599520,0.594317,0.589375,0.533913,0.555154,0.544326,0.634424,"[[307, 246], [268, 446]]","{'C': 0.1, 'class_weight': 'balanced'}"
6,Baseline,TF-IDF only,SVM (balanced),0.591121,0.590851,0.580336,0.591160,0.586878,0.529915,0.560579,0.544815,0.628940,"[[310, 243], [275, 439]]",NaN
11,Proposed,Chi-square,Random Forest,0.607477,0.599960,0.545126,0.602210,0.586758,0.552239,0.468354,0.506849,0.666667,"[[259, 294], [210, 504]]","{'max_depth': None, 'min_samples_split': 5, 'n..."
16,Proposed,Mutual Information,Random Forest,0.618380,0.609652,0.551282,0.607735,0.586362,0.565728,0.435805,0.492339,0.680386,"[[241, 312], [185, 529]]","{'max_depth': None, 'min_samples_split': 5, 'n..."
2,Baseline,TF-IDF only,Naive Bayes,0.611371,0.597663,0.523400,0.609313,0.580881,0.575521,0.399638,0.471718,0.690044,"[[221, 332], [163, 551]]",NaN


In [ ]:
final_results.to_csv("model_comparison_results_v2.csv", index=False)